In [6]:
import pandas as pd
import requests
import os
import yaml
from dotenv import load_dotenv

In [20]:
def request_config(url,param):
    load_dotenv()
    base_url = url

    headers = {'X-App-Token': os.environ.get("X-APP-TOKEN")}

    params = {'$query': param} 

    return base_url, params, headers

In [10]:
def get_data_response(url:str, subset, date, limit:int, offset:int):
    
    base_url, params, headers = request_config(url,f'SELECT * WHERE crash_date=\'{date}\' LIMIT {limit} OFFSET {offset}')
    
    response = requests.get(base_url, params=params, headers=headers)

    if response.status_code == 200:
        df = pd.json_normalize(response.json())

        file_path = f'./raw_data/{subset}/{date}_{subset}.csv'
        df.to_csv(file_path, index=False)

        print(f"Saved {len(df)} rows to {file_path}")
    else:
        print(f"Error - {response.status_code}, please check configuration!")

In [38]:
def get_date(url,query):
    base_url, earlist_params, headers = request_config(url,query)
    response = requests.get(base_url, params=earlist_params, headers=headers)

    df = pd.json_normalize(response.json())
    return pd.to_datetime(df['crash_date']).dt.date[0]

In [12]:
with open('./mvc.yaml', 'r') as f:
        mvc_data = yaml.load(f, Loader=yaml.FullLoader)
mvc_data

{'crashes': 'https://data.cityofnewyork.us/resource/h9gi-nx95.json',
 'vehicles': 'https://data.cityofnewyork.us/resource/bm4k-52h4.json',
 'persons': 'https://data.cityofnewyork.us/resource/f55k-p6yu.json'}

In [41]:
date = get_date('https://data.cityofnewyork.us/resource/h9gi-nx95.json','SELECT * ORDER BY crash_date ASC LIMIT 1')
date

datetime.date(2012, 7, 1)

In [42]:
get_data_response("https://data.cityofnewyork.us/resource/h9gi-nx95.json", 'crashes', date, 1000,0)

Saved 538 rows to ./raw_data/crashes/2012-07-01_crashes.csv
